# London Bike-Share Operations — Initial Validation & Demand Analysis

**Dataset:** TfL Cycle Hire journeys, October 2024  
**Business question:** Where and when should a bike-share operator prioritize fleet rebalancing to better match available bicycles with demand?

This notebook:
1. loads and validates the two October 2024 TfL journey extracts;
2. checks timestamp, duration, station, and boundary issues;
3. defines the final October analysis window;
4. profiles demand by day, hour, and station flow; and
5. exports validated datasets for downstream weather integration and BigQuery analysis.

> **Important:** journey arrivals and departures are used later as a proxy for rebalancing pressure. The source data does not contain live bike inventory, dock capacity, or operator rebalancing movements.


## 1. Setup and source files

The notebook assumes the project structure contains `data/raw/`, `data/processed/`, and `notebooks/`.  
The path logic below works whether the notebook is launched from the project root or from the `notebooks` folder.


In [1]:
from pathlib import Path

import pandas as pd

# Resolve the project root from either the root folder or notebooks/ folder.
cwd = Path.cwd()
project_dir = cwd.parent if cwd.name == "notebooks" else cwd

raw_dir = project_dir / "data" / "raw"
processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

file_1 = raw_dir / "405JourneyDataExtract01Oct2024-14Oct2024.csv"
file_2 = raw_dir / "406JourneyDataExtract15Oct2024-31Oct2024.csv"

# Confirm that both source files are available before continuing.
for path in [file_1, file_2]:
    print(
        path.name,
        "| exists:", path.exists(),
        "| size (MB):", round(path.stat().st_size / 1_000_000, 2) if path.exists() else None,
    )


405JourneyDataExtract01Oct2024-14Oct2024.csv | exists: True | size (MB): 59.14
406JourneyDataExtract15Oct2024-31Oct2024.csv | exists: True | size (MB): 79.18


In [2]:
# Load the two TfL journey extracts.
df_01_14_raw = pd.read_csv(file_1)
df_15_31_raw = pd.read_csv(file_2)

print("File 1 shape:", df_01_14_raw.shape)
print("File 2 shape:", df_15_31_raw.shape)

print("\nColumns:")
print(df_01_14_raw.columns.tolist())

display(df_01_14_raw.head())


File 1 shape: (355446, 11)
File 2 shape: (475738, 11)

Columns:
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']


,Number,Start date,Start station number,Start station,End date,End station number,End station,Bike number,Bike model,Total duration,Total duration (ms)
0,143694463,2024-10-14 23:59,1146,"Tysoe Street, Clerkenwell",2024-10-15 00:12,200152,"Pitfield Street North,Hoxton",52678,CLASSIC,12m 59s,779114
1,143694465,2024-10-14 23:59,200135,"Selby Street, Whitechapel",2024-10-15 00:05,200012,"Stepney Green Station, Stepney",57437,CLASSIC,5m 39s,339993
2,143694466,2024-10-14 23:59,300020,"West Kensington Station, West Kensington",2024-10-15 00:08,200221,"Crabtree Lane, Fulham",23059,CLASSIC,8m 55s,535381
3,143694467,2024-10-15 00:00,1077,"Royal London Hospital, Whitechapel",2024-10-15 00:15,200241,"Bonner Gate, Victoria Park",53678,CLASSIC,15m 17s,917475
4,143694460,2024-10-14 23:58,2687,"Brick Lane Market, Shoreditch",2024-10-15 00:31,200018,"Spindrift Avenue, Millwall",40299,CLASSIC,33m 6s,1986471


## 2. Basic data-quality checks

Before combining the extracts, validate row counts, missing values, and journey ID uniqueness in each file.


In [3]:
def profile_extract(df, label):
    # Print compact validation checks for one source extract.
    print(f"--- {label} ---")
    print("Rows:", len(df))
    print("Columns:", df.shape[1])
    print("Missing values:", int(df.isna().sum().sum()))
    print("Duplicate journey IDs:", int(df["Number"].duplicated().sum()))


profile_extract(df_01_14_raw, "Oct 1–14 extract")
profile_extract(df_15_31_raw, "Oct 15–31 extract")


--- Oct 1–14 extract ---
Rows: 355446
Columns: 11
Missing values: 0
Duplicate journey IDs: 0
--- Oct 15–31 extract ---
Rows: 475738
Columns: 11
Missing values: 0
Duplicate journey IDs: 0


## 3. Parse and validate timestamps

The source timestamps are local London clock times recorded only to the **minute**.  
Using an explicit format avoids ambiguous parsing behavior and keeps the transformation reproducible.


In [4]:
DATETIME_FORMAT = "%Y-%m-%d %H:%M"


def parse_journey_datetimes(df):
    # Copy the source data and parse the two journey timestamp columns.
    parsed = df.copy()

    parsed["Start date"] = pd.to_datetime(
        parsed["Start date"],
        format=DATETIME_FORMAT,
        errors="coerce",
    )

    parsed["End date"] = pd.to_datetime(
        parsed["End date"],
        format=DATETIME_FORMAT,
        errors="coerce",
    )

    return parsed


df_01_14 = parse_journey_datetimes(df_01_14_raw)
df_15_31 = parse_journey_datetimes(df_15_31_raw)

for label, df in [
    ("Oct 1–14 extract", df_01_14),
    ("Oct 15–31 extract", df_15_31),
]:
    print(f"--- {label} ---")
    print("Unparsed start timestamps:", df["Start date"].isna().sum())
    print("Unparsed end timestamps:", df["End date"].isna().sum())
    print("Earliest start:", df["Start date"].min())
    print("Latest start:", df["Start date"].max())


--- Oct 1–14 extract ---
Unparsed start timestamps: 0
Unparsed end timestamps: 0
Earliest start: 2024-10-01 00:00:00
Latest start: 2024-10-15 00:00:00
--- Oct 15–31 extract ---
Unparsed start timestamps: 0
Unparsed end timestamps: 0
Earliest start: 2024-10-15 00:00:00
Latest start: 2024-11-01 00:00:00


### Validate the boundary between the two files

The first extract contains one journey starting exactly at `2024-10-15 00:00`, while the second extract also begins at that clock minute.  
Journey IDs are therefore checked explicitly to ensure the extracts do not overlap.


In [5]:
# Check for journey IDs that appear in both extracts.
common_ids = set(df_01_14["Number"]).intersection(df_15_31["Number"])

print("Journey IDs appearing in both files:", len(common_ids))

print("\nLatest start timestamp in file 1:")
display(
    df_01_14.loc[
        df_01_14["Start date"] == df_01_14["Start date"].max(),
        ["Number", "Start date", "End date", "Start station", "End station"],
    ]
)

print("\nEarliest start timestamp in file 2:")
display(
    df_15_31.loc[
        df_15_31["Start date"] == df_15_31["Start date"].min(),
        ["Number", "Start date", "End date", "Start station", "End station"],
    ]
)


Journey IDs appearing in both files: 0

Latest start timestamp in file 1:


,Number,Start date,End date,Start station,End station
3,143694467,2024-10-15,2024-10-15 00:15:00,"Royal London Hospital, Whitechapel","Bonner Gate, Victoria Park"



Earliest start timestamp in file 2:


,Number,Start date,End date,Start station,End station
475735,143694468,2024-10-15,2024-10-15 00:14:00,"Pancras Road, King's Cross","Charlbert Street, St. John's Wood"
475736,143694469,2024-10-15,2024-10-15 00:06:00,"New Road 1 , Whitechapel","Stepney Green Station, Stepney"
475737,143694470,2024-10-15,2024-10-15 00:27:00,"New Cavendish Street, Marylebone","Warwick Road, Olympia"


## 4. Duration and daylight-saving validation

`Start date` and `End date` are minute-level local timestamps, while `Total duration (ms)` contains higher-precision elapsed duration.  
The two therefore should not be expected to match exactly.

A second issue occurs on **27 October 2024**, when UK clocks moved back one hour. Because the TfL timestamps are timezone-naive local clock values, some valid journeys appear to end before they start.


In [6]:
# Combine the extracts before validating duration behavior across the month.
df_october = pd.concat(
    [df_01_14, df_15_31],
    ignore_index=True,
)

# Calculate duration from the minute-level timestamps for comparison only.
timestamp_duration_ms = (
    (df_october["End date"] - df_october["Start date"])
    .dt.total_seconds()
    .mul(1000)
)

absolute_duration_difference_s = (
    timestamp_duration_ms
    .sub(df_october["Total duration (ms)"])
    .abs()
    .div(1000)
)

print("Combined rows:", len(df_october))
print("Duplicate journey IDs:", df_october["Number"].duplicated().sum())
print("Median absolute duration difference (s):", absolute_duration_difference_s.median())
print("Maximum absolute duration difference (s):", absolute_duration_difference_s.max())


Combined rows: 831184
Duplicate journey IDs: 0
Median absolute duration difference (s): 17.554
Maximum absolute duration difference (s): 3655.427


In [7]:
# Identify rows that appear to end before they start.
end_before_start_mask = df_october["End date"] < df_october["Start date"]
end_before_start_rows = df_october.loc[end_before_start_mask].copy()

print("Journeys with End date < Start date:", len(end_before_start_rows))
print("Earliest problematic start:", end_before_start_rows["Start date"].min())
print("Latest problematic start:", end_before_start_rows["Start date"].max())

display(
    end_before_start_rows[
        [
            "Number",
            "Start date",
            "End date",
            "Total duration",
            "Total duration (ms)",
            "Start station",
            "End station",
        ]
    ].head(10)
)


Journeys with End date < Start date: 71
Earliest problematic start: 2024-10-27 01:16:00
Latest problematic start: 2024-10-27 01:59:00


,Number,Start date,End date,Total duration,Total duration (ms),Start station,End station
498370,144035492,2024-10-27 01:59:00,2024-10-27 01:17:00,17m 47s,1067729,"Kennington Station, Kennington","South Lambeth Road, Vauxhall"
498371,144035493,2024-10-27 01:59:00,2024-10-27 01:12:00,12m 39s,759687,"Shadwell Station, Shadwell","Finsbury Leisure Centre, St. Luke's"
498372,144035494,2024-10-27 01:59:00,2024-10-27 01:09:00,10m 7s,607467,"York Hall, Bethnal Green","Harford Street, Mile End"
498373,144035496,2024-10-27 01:59:00,2024-10-27 01:13:00,13m 55s,835193,"Frith Street, Soho","Rodney Road , Walworth"
498375,144035486,2024-10-27 01:58:00,2024-10-27 01:02:00,4m 4s,244893,"Monier Road, Hackney Wick","Podium, Queen Elizabeth Olympic Park"
498377,144035488,2024-10-27 01:58:00,2024-10-27 01:31:00,33m 1s,1981365,"Vauxhall Street, Vauxhall","Ada Street, Hackney Central"
498379,144035490,2024-10-27 01:58:00,2024-10-27 01:07:00,8m 21s,501036,"Southampton Place, Holborn","Waterloo Road, South Bank"
498380,144035491,2024-10-27 01:58:00,2024-10-27 01:17:00,18m 5s,1085908,"Kennington Station, Kennington","South Lambeth Road, Vauxhall"
498385,144035478,2024-10-27 01:57:00,2024-10-27 01:10:00,13m 14s,794501,"Southampton Place, Holborn","Columbia Road, Shoreditch"
498387,144035481,2024-10-27 01:57:00,2024-10-27 01:14:00,16m 41s,1001541,"Longford Street, The Regent's Park","Wren Street, Holborn"


In [8]:
# Add one hour to the end timestamp for the DST rollback rows.
dst_adjusted_duration_ms = (
    (
        end_before_start_rows["End date"]
        + pd.Timedelta(hours=1)
        - end_before_start_rows["Start date"]
    )
    .dt.total_seconds()
    .mul(1000)
)

dst_duration_difference_s = (
    dst_adjusted_duration_ms
    .sub(end_before_start_rows["Total duration (ms)"])
    .abs()
    .div(1000)
)

print(dst_duration_difference_s.describe())
print(
    "DST rows matching recorded duration within 60 seconds:",
    (dst_duration_difference_s < 60).sum(),
    "of",
    len(dst_duration_difference_s),
)


count    71.000000
mean     20.361944
std      14.705663
min       0.874000
25%       7.988000
50%      14.501000
75%      31.569500
max      55.152000
dtype: float64
DST rows matching recorded duration within 60 seconds: 71 of 71


**Validation decision:** retain the DST-affected journeys and use **`Total duration (ms)` as the authoritative elapsed-duration field**. Timestamp-derived duration is used only as a validation check because the journey timestamps are minute-level and timezone-naive.


## 5. Define the October 2024 analysis window

The two source extracts include five journeys starting exactly at `2024-11-01 00:00`.  
The analytical dataset is therefore restricted to the half-open interval:

`2024-10-01 00:00 <= Start date < 2024-11-01 00:00`


In [9]:
analysis_start = pd.Timestamp("2024-10-01")
analysis_end = pd.Timestamp("2024-11-01")

df_analysis = df_october.loc[
    (df_october["Start date"] >= analysis_start)
    & (df_october["Start date"] < analysis_end)
].copy()

print("Analysis rows:", len(df_analysis))
print("Earliest start:", df_analysis["Start date"].min())
print("Latest start:", df_analysis["Start date"].max())
print("Unique start dates:", df_analysis["Start date"].dt.date.nunique())


Analysis rows: 831179
Earliest start: 2024-10-01 00:00:00
Latest start: 2024-10-31 23:59:00
Unique start dates: 31


## 6. Station, bike-model, and duration profiling


In [10]:
# Build a station ID-to-name lookup using both journey endpoints.
station_lookup = pd.concat(
    [
        df_analysis[["Start station number", "Start station"]]
        .rename(
            columns={
                "Start station number": "station_number",
                "Start station": "station_name",
            }
        ),
        df_analysis[["End station number", "End station"]]
        .rename(
            columns={
                "End station number": "station_number",
                "End station": "station_name",
            }
        ),
    ],
    ignore_index=True,
).drop_duplicates()

station_name_counts = (
    station_lookup.groupby("station_number")["station_name"].nunique()
)

print("Unique start stations:", df_analysis["Start station number"].nunique())
print("Unique end stations:", df_analysis["End station number"].nunique())
print(
    "Station IDs mapped to more than one name:",
    (station_name_counts > 1).sum(),
)

print("\nBike models:")
print(df_analysis["Bike model"].value_counts(dropna=False))


Unique start stations: 796
Unique end stations: 799
Station IDs mapped to more than one name: 0

Bike models:
Bike model
CLASSIC       701588
PBSC_EBIKE    129591
Name: count, dtype: int64


In [11]:
# Find stations that appear only on one side of customer journeys.
start_station_ids = set(df_analysis["Start station number"])
end_station_ids = set(df_analysis["End station number"])

start_only_ids = start_station_ids - end_station_ids
end_only_ids = end_station_ids - start_station_ids

print("Start-only stations:", len(start_only_ids))
print("End-only stations:", len(end_only_ids))

end_only_summary = (
    df_analysis.loc[df_analysis["End station number"].isin(end_only_ids)]
    .groupby(["End station number", "End station"])
    .size()
    .reset_index(name="trip_count")
    .sort_values("trip_count", ascending=False)
)

display(end_only_summary)

end_only_trip_count = end_only_summary["trip_count"].sum()
print(
    "Share of journeys ending at end-only stations:",
    f"{end_only_trip_count / len(df_analysis) * 100:.4f}%",
)


Start-only stations: 0
End-only stations: 3


,End station number,End station,trip_count
2,22168,Mechanical Workshop Clapham,49
0,10626,Mechanical Workshop Penton,6
1,22160,"Chelsea Bridge, Pimlico",1


Share of journeys ending at end-only stations: 0.0067%


The end-only locations are extremely rare. Mechanical workshop locations are retained in the validated data for completeness, but can be flagged or excluded from customer-facing rebalancing visuals because they are operational rather than normal customer stations.


In [12]:
# Create a numeric duration field from the authoritative millisecond column.
df_analysis["Duration minutes"] = (
    df_analysis["Total duration (ms)"] / 1000 / 60
)

duration_stats = df_analysis["Duration minutes"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)
display(duration_stats.to_frame("duration_minutes"))

# Summarize rare long-duration journeys without deleting them.
duration_thresholds = {
    ">24 hours": df_analysis["Duration minutes"] > 24 * 60,
    ">7 days": df_analysis["Duration minutes"] > 7 * 24 * 60,
    ">30 days": df_analysis["Duration minutes"] > 30 * 24 * 60,
}

duration_outliers = pd.DataFrame(
    [
        {
            "threshold": label,
            "journeys": int(mask.sum()),
            "share_pct": mask.mean() * 100,
        }
        for label, mask in duration_thresholds.items()
    ]
)

display(duration_outliers)


,duration_minutes
count,831179.000000
mean,28.833248
std,604.831436
min,0.015817
25%,7.700742
50%,12.992083
75%,20.435917
90%,29.251570
95%,37.312918
99%,75.610522


,threshold,journeys,share_pct
0,>24 hours,788,0.094805
1,>7 days,300,0.036093
2,>30 days,44,0.005294


**Duration decision:** retain long-duration records because they represent a very small share of the dataset and duration is not the main driver of the rebalancing analysis. Use robust statistics such as the median when describing journey duration.


## 7. Temporal demand patterns

Create reusable calendar fields and compare demand by weekday/weekend and hour of day.


In [13]:
# Add reusable calendar fields for demand analysis.
df_analysis["Date"] = df_analysis["Start date"].dt.date
df_analysis["Day of week"] = df_analysis["Start date"].dt.day_name()
df_analysis["Is weekend"] = df_analysis["Start date"].dt.dayofweek >= 5
df_analysis["Start hour"] = df_analysis["Start date"].dt.hour

day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]


In [14]:
# Calculate total and average daily starts by day of week.
starts_by_day = (
    df_analysis.groupby("Day of week")
    .size()
    .reindex(day_order)
    .rename("journey_starts")
)

calendar_days_by_weekday = (
    df_analysis[["Date", "Day of week"]]
    .drop_duplicates()
    .groupby("Day of week")
    .size()
    .reindex(day_order)
)

average_starts_by_day = (
    starts_by_day
    .div(calendar_days_by_weekday)
    .rename("avg_journey_starts_per_day")
    .reset_index()
)

display(average_starts_by_day)


,Day of week,avg_journey_starts_per_day
0,Monday,25129.0
1,Tuesday,27812.2
2,Wednesday,30685.0
3,Thursday,31087.8
4,Friday,27326.5
5,Saturday,23650.5
6,Sunday,19707.5


In [15]:
# Compare average daily demand for weekdays and weekends.
daily_totals = (
    df_analysis.groupby(["Date", "Is weekend"])
    .size()
    .reset_index(name="journey_starts")
)

weekday_weekend_avg = (
    daily_totals.groupby("Is weekend")["journey_starts"]
    .mean()
    .rename(index={False: "Weekday", True: "Weekend"})
    .reset_index(name="avg_journey_starts_per_day")
    .rename(columns={"Is weekend": "day_type"})
)

display(weekday_weekend_avg)

weekday_avg = weekday_weekend_avg.loc[
    weekday_weekend_avg["day_type"] == "Weekday",
    "avg_journey_starts_per_day",
].iloc[0]

weekend_avg = weekday_weekend_avg.loc[
    weekday_weekend_avg["day_type"] == "Weekend",
    "avg_journey_starts_per_day",
].iloc[0]

pct_higher = (weekday_avg - weekend_avg) / weekend_avg * 100
print(f"Average weekday demand is {pct_higher:.1f}% higher than weekend demand.")


,day_type,avg_journey_starts_per_day
0,Weekday,28597.695652
1,Weekend,21679.000000


Average weekday demand is 31.9% higher than weekend demand.


In [16]:
# Calculate average hourly starts separately for weekdays and weekends.
hourly_by_day_type = (
    df_analysis
    .groupby(["Date", "Is weekend", "Start hour"])
    .size()
    .reset_index(name="journey_starts")
    .groupby(["Is weekend", "Start hour"])["journey_starts"]
    .mean()
    .reset_index(name="avg_journey_starts_per_day")
)

hourly_pivot = (
    hourly_by_day_type
    .pivot(
        index="Start hour",
        columns="Is weekend",
        values="avg_journey_starts_per_day",
    )
    .rename(columns={False: "Weekday", True: "Weekend"})
)

display(hourly_pivot)


Is weekend,Weekday,Weekend
Start hour,,
0,153.347826,413.625
1,81.434783,315.500
2,47.782609,176.125
3,35.043478,131.625
4,51.652174,72.000
5,161.478261,77.500
6,621.695652,127.375
7,1764.173913,283.125
8,3514.956522,567.875


**Interpretation:** weekday demand has a pronounced morning and evening pattern consistent with commuting, while weekend demand is flatter and more concentrated around midday. Journey purpose is not observed, so the commute interpretation should be treated as a plausible pattern rather than a directly measured trip purpose.


## 8. Station flow and rebalancing-pressure proxy

For each station:

`net flow = arrivals - departures`

- Negative net flow indicates **depletion pressure**.
- Positive net flow indicates **accumulation pressure**.

This is a proxy only: actual bike inventory, dock capacity, and operator rebalancing movements are not available in the journey data.


In [17]:
# Calculate monthly departures and arrivals by station ID.
departures = (
    df_analysis.groupby("Start station number")
    .size()
    .rename("departures")
)

arrivals = (
    df_analysis.groupby("End station number")
    .size()
    .rename("arrivals")
)

station_flow = (
    pd.concat([departures, arrivals], axis=1)
    .fillna(0)
    .astype(int)
    .rename_axis("station_number")
    .reset_index()
)

station_flow["net_flow"] = (
    station_flow["arrivals"] - station_flow["departures"]
)

# Join the validated station name lookup.
station_flow = station_flow.merge(
    station_lookup.drop_duplicates("station_number"),
    on="station_number",
    how="left",
)

display(
    station_flow.sort_values("net_flow")
    [["station_number", "station_name", "departures", "arrivals", "net_flow"]]
    .head(10)
)

display(
    station_flow.sort_values("net_flow", ascending=False)
    [["station_number", "station_name", "departures", "arrivals", "net_flow"]]
    .head(10)
)


,station_number,station_name,departures,arrivals,net_flow
287,2692,"Waterloo Station 2, Waterloo",2352,1378,-974
73,1035,"Boston Place, Marylebone",1792,1354,-438
110,1075,"Hyde Park Corner, Hyde Park",4455,4041,-414
426,200017,"Lancaster Gate , Bayswater",1523,1117,-406
635,200246,"Eagle Wharf Road, Hoxton",2062,1691,-371
257,2586,"Cloudesley Road, Angel",922,581,-341
131,1096,"Bayswater Road, Hyde Park",1922,1589,-333
19,979,"Wenlock Road , Hoxton",1358,1039,-319
212,1183,"Bramham Gardens, Earl's Court",1797,1482,-315
250,1226,"Macclesfield Rd, St Lukes",1289,992,-297


,station_number,station_name,departures,arrivals,net_flow
22,982,"Holborn Circus, Holborn",1712,3139,1427
103,1067,"St. James's Square, St. James's",2518,3820,1302
38,999,"Queen Street 1, Bank",2275,3331,1056
192,1161,"Brushfield Street, Liverpool Street",3199,3920,721
1,960,"Hop Exchange, The Borough",3314,3962,648
405,22179,"Exhibition Road, Knightsbridge",3527,4163,636
90,1052,"Soho Square , Soho",2895,3507,612
617,200226,"Kingsway Southbound, Strand",1323,1890,567
127,1092,"Moorfields, Moorgate",2378,2940,562
244,1220,"Imperial College, Knightsbridge",2268,2812,544


In [18]:
# Restrict station-hour analysis to weekdays.
df_weekday = df_analysis.loc[~df_analysis["Is weekend"]].copy()
df_weekday["End hour"] = df_weekday["End date"].dt.hour

weekday_count = df_weekday["Date"].nunique()

hourly_departures = (
    df_weekday.groupby(["Start station number", "Start hour"])
    .size()
    .rename("departures")
    .rename_axis(["station_number", "hour"])
)

hourly_arrivals = (
    df_weekday.groupby(["End station number", "End hour"])
    .size()
    .rename("arrivals")
    .rename_axis(["station_number", "hour"])
)

hourly_station_flow = (
    pd.concat([hourly_departures, hourly_arrivals], axis=1)
    .fillna(0)
    .astype(int)
    .reset_index()
)

hourly_station_flow["net_flow"] = (
    hourly_station_flow["arrivals"] - hourly_station_flow["departures"]
)

hourly_station_flow["avg_net_flow_per_weekday"] = (
    hourly_station_flow["net_flow"] / weekday_count
)

hourly_station_flow = hourly_station_flow.merge(
    station_lookup.drop_duplicates("station_number"),
    on="station_number",
    how="left",
)

print("Number of weekdays:", weekday_count)
print("Station-hour rows:", len(hourly_station_flow))


Number of weekdays: 23
Station-hour rows: 18136


In [19]:
def show_peak_flow(hour, top_n=10):
    # Display the strongest depletion and accumulation pressure for one weekday hour.
    peak = hourly_station_flow.loc[
        hourly_station_flow["hour"] == hour,
        [
            "station_number",
            "station_name",
            "hour",
            "avg_net_flow_per_weekday",
        ],
    ]

    print(f"Weekday {hour:02d}:00 — strongest depletion")
    display(
        peak.sort_values("avg_net_flow_per_weekday")
        .head(top_n)
    )

    print(f"Weekday {hour:02d}:00 — strongest accumulation")
    display(
        peak.sort_values("avg_net_flow_per_weekday", ascending=False)
        .head(top_n)
    )


show_peak_flow(8)
show_peak_flow(17)


Weekday 08:00 — strongest depletion


,station_number,station_name,hour,avg_net_flow_per_weekday
2367,1072,"Waterloo Station 3, Waterloo",8,-98.347826
6335,2696,"Waterloo Station 1, Waterloo",8,-44.956522
1123,1011,"Argyle Street, Kings Cross",8,-36.130435
6244,2692,"Waterloo Station 2, Waterloo",8,-14.260870
13649,200246,"Eagle Wharf Road, Hoxton",8,-13.260870
16991,300252,"Canada Water Station, Rotherhithe",8,-11.782609
11914,200159,"Goldsmith's Row, Haggerston",8,-11.739130
1232,1017,"Great Dover Street, The Borough",8,-11.608696
1624,1035,"Boston Place, Marylebone",8,-11.043478
2898,1096,"Bayswater Road, Hyde Park",8,-10.782609


Weekday 08:00 — strongest accumulation


,station_number,station_name,hour,avg_net_flow_per_weekday
850,999,"Queen Street 1, Bank",8,35.521739
503,982,"Holborn Circus, Holborn",8,34.086957
2278,1067,"St. James's Square, St. James's",8,33.086957
2804,1092,"Moorfields, Moorgate",8,27.347826
4237,1161,"Brushfield Street, Liverpool Street",8,22.956522
8790,22180,"Cheapside, Bank",8,22.608696
10756,200094,"St. Bride Street, Holborn",8,21.304348
11301,200128,"Queen Street 2, Bank",8,20.521739
2985,1100,"Strand, Strand",8,17.695652
1299,1020,"Leonard Circus , Shoreditch",8,17.565217


Weekday 17:00 — strongest depletion


,station_number,station_name,hour,avg_net_flow_per_weekday
8799,22180,"Cheapside, Bank",17,-20.826087
2287,1067,"St. James's Square, St. James's",17,-17.608696
11310,200128,"Queen Street 2, Bank",17,-16.652174
9806,200044,"New Fetter Lane, Holborn",17,-14.217391
5059,1204,"Queen's Gate (North), Kensington",17,-14.086957
2420,1074,"Newgate Street , St. Paul's",17,-13.869565
859,999,"Queen Street 1, Bank",17,-13.347826
2813,1092,"Moorfields, Moorgate",17,-13.217391
10765,200094,"St. Bride Street, Holborn",17,-12.956522
815,997,"Worship Street, Shoreditch",17,-12.391304


Weekday 17:00 — strongest accumulation


,station_number,station_name,hour,avg_net_flow_per_weekday
2376,1072,"Waterloo Station 3, Waterloo",17,99.217391
6344,2696,"Waterloo Station 1, Waterloo",17,26.652174
1132,1011,"Argyle Street, Kings Cross",17,26.304348
6253,2692,"Waterloo Station 2, Waterloo",17,9.652174
4724,1186,"Belvedere Road 1, South Bank",17,7.347826
971,1004,"Doric Way , Somers Town",17,6.652174
16116,300207,"Gauden Road, Clapham",17,5.130435
1633,1035,"Boston Place, Marylebone",17,4.869565
17069,1190444,"Kennington Lane Rail Bridge, Vauxhall_OLD",17,4.739130
4553,1176,"Waterloo Road, South Bank",17,4.695652


In [20]:
# Compare each station's average net flow at the 08:00 and 17:00 weekday peaks.
morning_flow = (
    hourly_station_flow.loc[
        hourly_station_flow["hour"] == 8,
        ["station_number", "station_name", "avg_net_flow_per_weekday"],
    ]
    .rename(columns={"avg_net_flow_per_weekday": "avg_net_flow_08"})
)

evening_flow = (
    hourly_station_flow.loc[
        hourly_station_flow["hour"] == 17,
        ["station_number", "station_name", "avg_net_flow_per_weekday"],
    ]
    .rename(columns={"avg_net_flow_per_weekday": "avg_net_flow_17"})
)

peak_reversal = (
    morning_flow.merge(
        evening_flow,
        on=["station_number", "station_name"],
        how="outer",
    )
    .fillna(0)
)

peak_reversal["true_reversal"] = (
    peak_reversal["avg_net_flow_08"]
    * peak_reversal["avg_net_flow_17"]
    < 0
)

peak_reversal["reversal_magnitude"] = (
    peak_reversal["avg_net_flow_17"]
    - peak_reversal["avg_net_flow_08"]
).abs()

peak_reversal["peak_pressure"] = (
    peak_reversal["avg_net_flow_08"].abs()
    + peak_reversal["avg_net_flow_17"].abs()
)

print(
    "Stations with true morning-evening reversal:",
    int(peak_reversal["true_reversal"].sum()),
)

display(
    peak_reversal.sort_values("peak_pressure", ascending=False).head(20)
)


Stations with true morning-evening reversal: 661


,station_number,station_name,avg_net_flow_08,avg_net_flow_17,true_reversal,reversal_magnitude,peak_pressure
107,1072,"Waterloo Station 3, Waterloo",-98.347826,99.217391,True,197.565217,197.565217
291,2696,"Waterloo Station 1, Waterloo",-44.956522,26.652174,True,71.608696,71.608696
50,1011,"Argyle Street, Kings Cross",-36.130435,26.304348,True,62.434783,62.434783
103,1067,"St. James's Square, St. James's",33.086957,-17.608696,True,50.695652,50.695652
38,999,"Queen Street 1, Bank",35.521739,-13.347826,True,48.869565,48.869565
22,982,"Holborn Circus, Holborn",34.086957,-11.695652,True,45.782609,45.782609
408,22180,"Cheapside, Bank",22.608696,-20.826087,True,43.434783,43.434783
127,1092,"Moorfields, Moorgate",27.347826,-13.217391,True,40.565217,40.565217
526,200128,"Queen Street 2, Bank",20.521739,-16.652174,True,37.173913,37.173913
500,200094,"St. Bride Street, Holborn",21.304348,-12.956522,True,34.260870,34.260870


In [21]:
# Find the single weekday hour with the strongest absolute net-flow pressure for each station.
hourly_station_flow["abs_avg_net_flow"] = (
    hourly_station_flow["avg_net_flow_per_weekday"].abs()
)

max_pressure_idx = (
    hourly_station_flow.groupby("station_number")["abs_avg_net_flow"]
    .idxmax()
)

station_peak_pressure = (
    hourly_station_flow.loc[
        max_pressure_idx,
        [
            "station_number",
            "station_name",
            "hour",
            "avg_net_flow_per_weekday",
            "abs_avg_net_flow",
        ],
    ]
    .sort_values("abs_avg_net_flow", ascending=False)
)

display(station_peak_pressure.head(20))


,station_number,station_name,hour,avg_net_flow_per_weekday,abs_avg_net_flow
2376,1072,"Waterloo Station 3, Waterloo",17,99.217391,99.217391
6335,2696,"Waterloo Station 1, Waterloo",8,-44.956522,44.956522
1123,1011,"Argyle Street, Kings Cross",8,-36.130435,36.130435
850,999,"Queen Street 1, Bank",8,35.521739,35.521739
503,982,"Holborn Circus, Holborn",8,34.086957,34.086957
2278,1067,"St. James's Square, St. James's",8,33.086957,33.086957
2804,1092,"Moorfields, Moorgate",8,27.347826,27.347826
4237,1161,"Brushfield Street, Liverpool Street",8,22.956522,22.956522
8790,22180,"Cheapside, Bank",8,22.608696,22.608696
6243,2692,"Waterloo Station 2, Waterloo",7,-22.608696,22.608696


## 9. Export validated datasets

Two outputs are created:

- `tfl_october_2024_validated.parquet` — validated October journey data used in the Python workflow.
- `journeys_october_2024_bigquery.parquet` — a SQL-friendly version with snake_case field names for BigQuery.


In [ ]:
# Save the validated October analysis dataset.
processed_bike_path = processed_dir / "tfl_october_2024_validated.parquet"

df_analysis.to_parquet(
    processed_bike_path,
    index=False,
)

print("Saved:", processed_bike_path)
print("Rows saved:", len(df_analysis))


Saved: /Users/erik/Documents/Projects/bike-share-operations/data/processed/tfl_october_2024_validated_test.parquet
Rows saved: 831179


In [ ]:
# Keep only the fields needed in the BigQuery analytical journey table.
journeys_bigquery = df_analysis[
    [
        "Number",
        "Start date",
        "Start station number",
        "Start station",
        "End date",
        "End station number",
        "End station",
        "Bike number",
        "Bike model",
        "Total duration",
        "Total duration (ms)",
    ]
].copy()

# Rename columns to SQL-friendly snake_case names.
journeys_bigquery = journeys_bigquery.rename(
    columns={
        "Number": "journey_id",
        "Start date": "start_datetime",
        "Start station number": "start_station_id",
        "Start station": "start_station_name",
        "End date": "end_datetime",
        "End station number": "end_station_id",
        "End station": "end_station_name",
        "Bike number": "bike_id",
        "Bike model": "bike_model",
        "Total duration": "duration_text",
        "Total duration (ms)": "duration_ms",
    }
)

# Add a convenient numeric duration field for SQL analysis.
journeys_bigquery["duration_minutes"] = (
    journeys_bigquery["duration_ms"] / 1000 / 60
)

journeys_bq_path = (
    processed_dir / "journeys_october_2024_bigquery.parquet"
)

journeys_bigquery.to_parquet(
    journeys_bq_path,
    index=False,
)

print("Rows:", len(journeys_bigquery))
print("Columns:", journeys_bigquery.columns.tolist())
print("Saved:", journeys_bq_path)


Rows: 831179
Columns: ['journey_id', 'start_datetime', 'start_station_id', 'start_station_name', 'end_datetime', 'end_station_id', 'end_station_name', 'bike_id', 'bike_model', 'duration_text', 'duration_ms', 'duration_minutes']
Saved: /Users/erik/Documents/Projects/bike-share-operations/data/processed/journeys_october_2024_bigquery_test.parquet


## 10. Key validation decisions

- The final October analysis window contains journeys starting from **1 October through 31 October 2024**.
- `Total duration (ms)` is used as the authoritative elapsed-duration field because timestamps are minute-level and DST-sensitive.
- The 27 October daylight-saving rollback records are retained rather than treated as invalid journeys.
- Rare long-duration journeys are retained because they represent a very small share of records and duration is not central to the rebalancing question.
- Mechanical workshop destinations are retained in the underlying data but should be excluded or flagged in customer-facing station-pressure visuals.
- Station net flow is interpreted as a **rebalancing-pressure proxy**, not direct evidence of empty or full docks.
